# Module 08 - Multi-Head Attention Notebook

This notebook extends Module 07's single-head attention into multi-head attention. The main things to make physical are the reshape, the `sqrt(head_dim)` scaling, matched-parameter head comparisons, and per-head heatmaps.

In [ ]:
from __future__ import annotations

import contextlib
import io
import math
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

import g2c
from g2c.attention import MultiHeadAttention
from g2c.embeddings import LearnedPositionalEmbedding, TokenEmbedding
from g2c.nn import CrossEntropyLoss, Linear, Module, SGD, resolve_device
from g2c.tokenizer import BPETokenizer

_ = torch.manual_seed(0)
repo_root = Path(g2c.__file__).resolve().parents[1]
experiment_device = "auto"
print("MPS available:", torch.backends.mps.is_available())

## Before the Notebook

Implement `MultiHeadAttention.forward` and `MultiHeadAttention.attention_weights` first. The notebook assumes the Module 08 tests are green.

In [ ]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_multi_head_attention.py -x"
"Question: Which multi-head test is the next one failing, and what implementation detail does it point at?"
"Answer: "

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_multi_head_attention.py"],
    cwd=repo_root,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

assert result.returncode == 0, "Module 08 multi-head attention tests are not passing yet."
print("Module 08 multi-head attention tests passed.")

## Exercise 1 - Verify Per-Head Scaling by Hand

Force Q and K projections to the identity so each head sees a clean slice of the input. Then compare `MultiHeadAttention.attention_weights` to a manual per-head softmax over `x_h @ x_h.T / sqrt(head_dim)`.

In [ ]:
"Question: With D=4 and H=2, what is head_dim?"
"Answer: "

"Question: Why is the scale sqrt(head_dim), not sqrt(D)?"
"Answer: "

In [ ]:
D, H = 4, 2
head_dim = D // H
mha = MultiHeadAttention(embedding_dim=D, num_heads=H, causal=False)
with torch.no_grad():
    mha.q_proj.W.copy_(torch.eye(D))
    mha.q_proj.b.zero_()
    mha.k_proj.W.copy_(torch.eye(D))
    mha.k_proj.b.zero_()

x = torch.tensor([[[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0]]])
actual = mha.attention_weights(x)
x_per_head = x.view(1, 2, H, head_dim).transpose(1, 2)
expected_scores = x_per_head @ x_per_head.transpose(-2, -1) / math.sqrt(head_dim)
expected = expected_scores.softmax(dim=-1)

print("actual weights shape:", tuple(actual.shape))
print("head 0 weights:")
print(actual[0, 0].detach())
print("head 1 weights:")
print(actual[0, 1].detach())
print("max absolute diff:", (actual - expected).abs().max().item())

assert torch.allclose(actual, expected, atol=1e-5)

## Exercise 2 - Reshape Order Matters

Both reshape orders below produce tensors with valid-looking shapes. They do not assign channels to heads the same way. Trace the numbers and explain which one matches the canonical implementation.

In [ ]:
"Question: What channels should head 0 receive when D=8, H=4, head_dim=2?"
"Answer: "

"Question: Why can a wrong reshape preserve shapes but still change the model?"
"Answer: "

In [ ]:
B, T, D, H = 1, 1, 8, 4
head_dim = D // H
q = torch.arange(D).float().view(B, T, D)

canonical = q.view(B, T, H, head_dim).transpose(1, 2)
swapped = q.view(B, T, head_dim, H).transpose(1, 2)

print("original:", q[0, 0].tolist())
for h in range(H):
    print(f"canonical head {h}:", canonical[0, h, 0].tolist())
print()
for h in range(head_dim):
    print(f"swapped axis {h}:", swapped[0, h, 0].tolist())

## Exercise 3 - Train Tiny LMs at H = 1, 4, 8

These models keep `embedding_dim` fixed, so the attention parameter count is the same. If validation curves differ, the difference comes from the structure of the computation, not from adding more attention parameters.

In [ ]:
"Question: Why is this a fairer comparison than changing D and H together?"
"Answer: "

"Question: What failure mode might appear if head_dim becomes too small?"
"Answer: "

In [ ]:
def load_mha_text(max_chars: int = 20_000) -> str:
    path = repo_root / "data" / "tinyshakespeare.txt"
    if path.exists():
        return path.read_text(encoding="utf-8")[:max_chars]
    base = """
    the model predicts the next token from context
    attention lets tokens choose which earlier tokens matter
    multiple heads can look for different local patterns
    the same projection size can be split into different head counts
    """
    return ("\n".join(line.strip() for line in base.strip().splitlines()) + "\n") * 200


mha_text = load_mha_text()
mha_tokenizer = BPETokenizer()
with contextlib.redirect_stdout(io.StringIO()):
    mha_tokenizer.train(mha_text, vocab_size=384)
mha_ids = torch.tensor(mha_tokenizer.encode(mha_text), dtype=torch.long)
split = int(0.9 * len(mha_ids))
mha_train_ids = mha_ids[:split]
mha_val_ids = mha_ids[split:]
mha_vocab_size = len(mha_tokenizer.vocab)

print("tokens:", len(mha_ids))
print("vocab size:", mha_vocab_size)
print("train tokens:", len(mha_train_ids))
print("val tokens:", len(mha_val_ids))


class TinyMultiHeadLM(Module):
    def __init__(self, vocab_size: int, seq_len: int, embedding_dim: int, num_heads: int) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.embed = TokenEmbedding(vocab_size, embedding_dim)
        self.pos = LearnedPositionalEmbedding(seq_len, embedding_dim)
        self.attn = MultiHeadAttention(embedding_dim, num_heads, causal=True)
        self.out = Linear(embedding_dim, vocab_size)

    def parameters(self):
        return [
            *self.embed.parameters(),
            *self.pos.parameters(),
            *self.attn.parameters(),
            *self.out.parameters(),
        ]

    def hidden_states(self, ids: torch.Tensor) -> torch.Tensor:
        B, T = ids.shape
        return self.embed(ids) + self.pos(T).unsqueeze(0)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        x = self.hidden_states(ids)
        x = self.attn(x)
        return self.out(x)

In [ ]:
def get_sequence_batch(ids: torch.Tensor, seq_len: int, batch_size: int, *, generator=None):
    starts = torch.randint(0, len(ids) - seq_len - 1, (batch_size,), generator=generator)
    x = torch.stack([ids[s : s + seq_len] for s in starts])
    y = torch.stack([ids[s + 1 : s + seq_len + 1] for s in starts])
    return x, y


def sequence_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return CrossEntropyLoss()(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))


def estimate_perplexity(model, ids, *, batches=8, batch_size=64, seed=123, device=None) -> float:
    if device is None:
        device = getattr(model, "device", torch.device("cpu"))
    else:
        device = resolve_device(device)
    generator = torch.Generator().manual_seed(seed)
    losses = []
    with torch.no_grad():
        for _ in range(batches):
            x, y = get_sequence_batch(ids, model.seq_len, batch_size, generator=generator)
            x = x.to(device)
            y = y.to(device)
            losses.append(float(sequence_loss(model(x), y).item()))
    return math.exp(sum(losses) / len(losses))


def train_mha_lm(num_heads: int, *, num_steps=300, log_every=50, seed=0, device=experiment_device):
    device = resolve_device(device)
    model = TinyMultiHeadLM(
        vocab_size=mha_vocab_size,
        seq_len=16,
        embedding_dim=64,
        num_heads=num_heads,
    ).to(device)
    optimizer = SGD(model.parameters(), lr=0.15)
    generator = torch.Generator().manual_seed(seed)
    val_perplexities = []
    train_losses = []
    steps = []
    for step in range(num_steps):
        x, y = get_sequence_batch(mha_train_ids, model.seq_len, 64, generator=generator)
        x = x.to(device)
        y = y.to(device)
        loss = sequence_loss(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % log_every == 0 or step == num_steps - 1:
            steps.append(step)
            train_losses.append(float(loss.item()))
            val_perplexities.append(estimate_perplexity(model, mha_val_ids, seed=seed + step + 1, device=device))
    return model, {"steps": steps, "train_losses": train_losses, "val_perplexities": val_perplexities}

In [ ]:
head_counts = [1, 4, 8]
trained_models = {}
histories = {}

for i, num_heads in enumerate(head_counts):
    print(f"training H={num_heads}")
    model, history = train_mha_lm(num_heads, seed=10 + i)
    trained_models[num_heads] = model
    histories[num_heads] = history
    print("  final val perplexity:", history["val_perplexities"][-1])

plt.figure(figsize=(7, 4))
for num_heads in head_counts:
    history = histories[num_heads]
    plt.plot(history["steps"], history["val_perplexities"], marker="o", label=f"H={num_heads}")
plt.yscale("log")
plt.title("Validation perplexity at fixed D=64")
plt.xlabel("step")
plt.ylabel("perplexity")
plt.legend()
plt.show()

## Exercise 4 - Visualize Per-Head Attention

Pick one trained model and plot one heatmap per head. At this scale the patterns may be partial or noisy. The important thing is that `attention_weights` exposes `(B, H, T, T)`, not an average over heads.

In [ ]:
"Question: Does any head look like a previous-token head? What visual pattern would that be?"
"Answer: "

"Question: Why would averaging over heads hide the point of this visualization?"
"Answer: "

In [ ]:
def label_piece(token_id: int) -> str:
    piece = mha_tokenizer.decode([int(token_id)]).replace("\n", "\\n")
    if piece == " ":
        return "space"
    return piece


viz_model = trained_models[4]
viz_text = "First Citizen:\nBefore we proceed any further"
viz_ids = torch.tensor(mha_tokenizer.encode(viz_text), dtype=torch.long)[: viz_model.seq_len]
viz_batch = viz_ids.unsqueeze(0).to(viz_model.device)
labels = [f"{i}:{label_piece(token_id)}" for i, token_id in enumerate(viz_ids.tolist())]

x = viz_model.hidden_states(viz_batch)
weights = viz_model.attn.attention_weights(x)[0].detach().cpu()

cols = 2
rows = math.ceil(viz_model.num_heads / cols)
fig, axes = plt.subplots(rows, cols, figsize=(10, 4 * rows), constrained_layout=True)
axes = axes.reshape(-1)
for h in range(viz_model.num_heads):
    ax = axes[h]
    image = ax.imshow(weights[h], vmin=0.0, vmax=float(weights[h].max()), cmap="magma")
    ax.set_title(f"head {h}")
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
for ax in axes[viz_model.num_heads:]:
    ax.axis("off")
plt.show()

### Baseline-Adjusted Attention

The raw heatmap often mostly shows the causal mask. The next plot subtracts the baseline pattern "uniform attention over all allowed previous positions." Red means the head attends more than that baseline; blue means less. If the adjusted plot is nearly white, the head is behaving close to causal-uniform attention.

In [ ]:
def causal_uniform_baseline(seq_len: int, *, device=None, dtype=None) -> torch.Tensor:
    allowed = torch.tril(torch.ones(seq_len, seq_len, device=device, dtype=dtype))
    return allowed / allowed.sum(dim=-1, keepdim=True)


baseline = causal_uniform_baseline(
    weights.shape[-1],
    device=weights.device,
    dtype=weights.dtype,
)
centered_weights = weights - baseline
allowed_positions = torch.tril(
    torch.ones_like(baseline, dtype=torch.bool)
)
allowed_positions[0, 0] = False

print("deviation from causal-uniform baseline")
print("-" * 48)
for h in range(viz_model.num_heads):
    diffs = centered_weights[h][allowed_positions].abs()
    print(
        f"head {h}: mean_abs_diff={diffs.mean().item():.4f}  "
        f"max_abs_diff={diffs.max().item():.4f}"
    )

max_abs = max(float(centered_weights.abs().max().item()), 1e-6)
fig, axes = plt.subplots(rows, cols, figsize=(10, 4 * rows), constrained_layout=True)
axes = axes.reshape(-1)
for h in range(viz_model.num_heads):
    ax = axes[h]
    image = ax.imshow(
        centered_weights[h],
        vmin=-max_abs,
        vmax=max_abs,
        cmap="coolwarm",
    )
    ax.set_title(f"head {h} minus causal-uniform")
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
for ax in axes[viz_model.num_heads:]:
    ax.axis("off")
plt.show()

In [ ]:
"Question: Which head deviates most from the causal-uniform baseline?"
"Answer: "

"Question: If all adjusted heatmaps are nearly white, what does that say about this tiny model's head specialization?"
"Answer: "

## Exercise 5 - Parameter Counts at Varying H

The number of heads changes the internal structure but not the total attention parameter count when `D` is fixed.

In [ ]:
"Question: If the parameter count is identical, why can different H values behave differently?"
"Answer: "

In [ ]:
print("D    H    head_dim    expected    actual")
print("-" * 48)
for D, H in [(64, 1), (64, 4), (64, 8), (128, 4), (128, 8)]:
    attn = MultiHeadAttention(embedding_dim=D, num_heads=H)
    expected = 4 * (D * D + D)
    actual = sum(p.numel() for p in attn.parameters())
    print(f"{D:<4d} {H:<4d} {D // H:<10d} {expected:<10d} {actual:<10d}")
    assert expected == actual